In [1]:
!module load python


The following have been reloaded with a version change:
  1) python/3.9.6-gc563 => python/3.13.9-ij92



In [2]:
!pip install qepy qiskit qiskit-nature numpy scipy jupyter

In [3]:
!pip install "qiskit>=1.1" "qiskit-algorithms>=0.3" "qiskit-nature>=0.7"
import sys, subprocess

In [4]:
!export W90_BIN="/home/am4655/q-e/bin/wannier90.x"
!export PW2W90_BIN="/home/am4655/q-e/bin/pw2wannier90.x"

In [16]:
# QEpy → Wannier90 → Qiskit (QEpy-only, venv-friendly)

from __future__ import annotations
import os, shutil, subprocess, re
from pathlib import Path
from dataclasses import dataclass
from typing import List, Tuple, Dict
import numpy as np

from qepy.driver import Driver

from qiskit_algorithms.optimizers import SPSA
from qiskit.circuit.library import EfficientSU2
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives import Estimator
from qiskit_algorithms import VQE
from qiskit_nature.second_q.operators import FermionicOp
from qiskit_nature.second_q.mappers import JordanWignerMapper

def run(cmd: List[str], cwd: Path):
    print("[run]", " ".join(cmd))
    res = subprocess.run(" ".join(cmd), cwd=str(cwd), shell=True, capture_output=True, text=True)
    print(res.stdout)
    if res.returncode != 0:
        print(res.stderr)
        raise RuntimeError(f"Command failed: {' '.join(cmd)}")

def find_exec(candidates: list[str]) -> str:
    """Return first matching executable in PATH from candidates."""
    import shutil as _sh
    for name in candidates:
        path = _sh.which(name)
        if path:
            return path
    raise RuntimeError(f"None of {candidates} found in PATH.")

# Prefer explicit env vars (great for venv setups), then fall back to PATH
W90_BIN    = os.environ.get("W90_BIN")    or find_exec(["wannier90.x", "wannier90"])
PW2W90_BIN = os.environ.get("PW2W90_BIN") or find_exec(["pw2wannier90.x"])

print("[info] Using:", W90_BIN, "and", PW2W90_BIN)


[info] Using: /home/am4655/q-e/bin/wannier90.x and /home/am4655/q-e/bin/pw2wannier90.x


In [17]:
import qepy, inspect
from qepy.io import QEInput

In [7]:
from pathlib import Path
import os
from qepy.driver import Driver  # IMPORTANT: import from qepy.driver

# --- config you change ---
qe_input = Path("CH4.scf.in")      # QE input in this folder
workdir  = qe_input.parent.resolve()
logfile  = workdir / "qepy.log"
scf_out  = workdir / "scf.out"
# --------------------------

# ensure correct CWD & input exists
os.chdir(workdir)
assert qe_input.exists(), f"Input file not found: {qe_input}"

# Driver expects an INPUT FILE PATH (string), not a QEInput object.
drv = Driver(str(qe_input), comm=None, logfile=str(logfile))


scf_res = drv.scf()

# Persist stdout if available
out = getattr(scf_res, "stdout", None) or "[QEpy completed SCF]\n"
scf_out.write_text(out)

print(f"[ok] SCF finished.\n- log: {logfile}\n- out: {scf_out}\n- cwd: {Path.cwd()}")
print("Energy:\n", drv.get_energy())

[ok] SCF finished.
- log: /cache/home/am4655/qepy_eg/QCOMP/qepy.log
- out: /cache/home/am4655/qepy_eg/QCOMP/scf.out
- cwd: /cache/home/am4655/qepy_eg/QCOMP
Energy:
 -17.187786281017402


In [10]:
print(dir(drv))

['FORCENAMES', 'POTNAMES', 'STRESSNAMES', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattr__', '__getattribute__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_comm', '_commf', '_embed', '_init_log', 'atoms', 'calc_energy', 'check_convergence', 'comm', 'commf', 'create_array', 'data2field', 'density', 'diagonalize', 'driver_initialize', 'electrons', 'embed', 'end_scf', 'field2data', 'fileobj', 'fileobj_interact', 'forcename2type', 'get_ase_atoms', 'get_bz_k_points', 'get_core_density', 'get_density', 'get_density_functional', 'get_density_functional_potential', 'get_dftpy_grid', 'get_dftpy_ions', 'get_dipole_tddft', 'get_ecutrho', 'get_effective_potential', 'get_eigenvalues', 'get_elf', 'get_energy', 'get_exchange_correlation', 'get_exchange_corre

In [18]:
import os, re, shutil, subprocess, textwrap, math

# --- config from your session ---
# Ensure these exist (adjust if needed)
workdir   = Path(".").resolve()
qe_input  = next((p for p in [workdir/"CH4.scf.in", workdir/"scf.in", workdir/"pw.in"] if p.exists()), None)
assert qe_input is not None, "QE input not found near scf.out"
# Parse prefix from input if not explicitly set
def parse_prefix_from_in(qe_in: Path, default_prefix="pwscf"):
    txt = qe_in.read_text(errors="ignore")
    m = re.search(r"&control(.*?)/", txt, re.I | re.S)
    block = m.group(1) if m else ""
    m2 = re.search(r"\bprefix\s*=\s*'?([^',\s]+)'?", block, re.I)
    return (m2.group(1).strip() if m2 else default_prefix)
prefix = parse_prefix_from_in(qe_input)
seedname = prefix  # keep them the same for a smooth handoff

In [19]:
# --- helpers ---
def which_or_raise(name, alt=None):
    p = shutil.which(name) or (shutil.which(alt) if alt else None)
    if not p:
        raise FileNotFoundError(f"Cannot find '{name}'" + (f" or '{alt}'" if alt else "") + " in PATH.")
    return p

def run_cmd(cmd, *, cwd=None, stdin_text=None, check=True):
    cp = subprocess.run(cmd, cwd=cwd, input=stdin_text, text=True,
                        capture_output=True, check=check)
    return cp

def parse_outdir_prefix(qe_in: Path, default_outdir="./"):
    txt = qe_in.read_text(errors="ignore")
    ctrl = re.search(r"&control(.*?)/", txt, re.I | re.S)
    block = ctrl.group(1) if ctrl else ""
    def grab(key, default):
        m = re.search(rf"\b{key}\s*=\s*'?([^',\n]+)'?", block, re.I)
        return m.group(1).strip() if m else default
    return grab("outdir", default_outdir), grab("prefix", prefix)

from pathlib import Path
from typing import Tuple

def check_prefix_save(outdir: Path, prefix: str):
    save = outdir / f"{prefix}.save"
    print(f"[check] looking for {save}")
    return save.exists()

In [20]:
from pathlib import Path
import re
from typing import Iterable, Optional, Tuple, List
from collections import Counter

BOHR_TO_ANG = 0.52917721092

def _block_after(header_regex: str, text: str):
    m = re.search(header_regex, text, flags=re.I|re.M)
    if not m: return None
    rest = text[m.end():]
    stop = re.search(r"^\s*(?:&\w+|K_POINTS|ATOMIC_|CELL_|ELECTRONS|IONS|SYSTEM|CONTROL)\b",
                     rest, flags=re.I|re.M)
    return rest[:stop.start()] if stop else rest

def _grab_system_val(text: str, key: str, default=None):
    m = re.search(r"&system(.*?)/", text, re.I|re.S)
    blk = m.group(1) if m else ""
    m2 = re.search(rf"\b{re.escape(key)}\s*=\s*('?)([^,\n]+)\1", blk, re.I)
    if not m2: return default
    raw = m2.group(2).strip()
    try:
        return float(raw) if any(c in raw for c in ".eE") else int(raw)
    except Exception:
        return raw

# ---------- geometry parsing ----------
def _parse_cell_from_ibrav1(text: str):
    ibrav = int(_grab_system_val(text, "ibrav", 0) or 0)
    a_bohr = _grab_system_val(text, "celldm(1)")
    if ibrav != 1 or a_bohr is None:
        raise RuntimeError("Need CELL_PARAMETERS or (ibrav=1 & celldm(1)).")
    a_ang = float(a_bohr) * BOHR_TO_ANG
    return [[a_ang,0.0,0.0],[0.0,a_ang,0.0],[0.0,0.0,a_ang]]

def _parse_atoms(text: str, lattice_ang):
    hdr = re.search(r"(?im)^\s*ATOMIC_POSITIONS\s*\{([^}]+)\}\s*$", text)
    if not hdr: raise RuntimeError("ATOMIC_POSITIONS block not found.")
    unit = hdr.group(1).strip().lower()
    blk = _block_after(r"(?im)^\s*ATOMIC_POSITIONS\s*\{[^}]+\}\s*$", text)
    lines = [ln.strip() for ln in blk.strip().splitlines() if ln.strip()]
    atoms = []
    for ln in lines:
        toks = ln.split()
        sym = toks[0]; x,y,z = map(float, toks[1:4])
        atoms.append((sym,x,y,z))
    if "angstrom" in unit:
        return atoms
    if "bohr" in unit:
        return [(s, x*BOHR_TO_ANG, y*BOHR_TO_ANG, z*BOHR_TO_ANG) for s,x,y,z in atoms]
    if "alat" in unit:
        a_bohr = _grab_system_val(text, "celldm(1)")
        if a_bohr is None: raise RuntimeError("ATOMIC_POSITIONS {alat} but celldm(1) missing.")
        a_ang = float(a_bohr) * BOHR_TO_ANG
        return [(s, x*a_ang, y*a_ang, z*a_ang) for s,x,y,z in atoms]
    if "crystal" in unit:
        a1,a2,a3 = lattice_ang
        def f2c(fx,fy,fz):
            return (fx*a1[0]+fy*a2[0]+fz*a3[0],
                    fx*a1[1]+fy*a2[1]+fz*a3[1],
                    fx*a1[2]+fy*a2[2]+fz*a3[2])
        return [(s,)+f2c(x,y,z) for s,x,y,z in atoms]
    raise RuntimeError(f"Unknown ATOMIC_POSITIONS unit: {unit}")

# ---------- k-point parsing/generation ----------
def _parse_explicit_kpoints(text: str) -> Optional[List[Tuple[float,float,float,int]]]:
    # K_POINTS {crystal|crystal_b|crystal_w}
    m = re.search(r"(?im)^\s*K_POINTS\s*\{\s*crystal(?:_b|_w)?\s*\}\s*$", text)
    if not m:
        return None
    blk = _block_after(r"(?im)^\s*K_POINTS\s*\{\s*crystal(?:_b|_w)?\s*\}\s*$", text)
    if not blk: return None
    lines = [ln.strip() for ln in blk.strip().splitlines() if ln.strip()]

    kpts: List[Tuple[float,float,float,int]] = []
    start_idx = 0
    try:
        ndecl = int(lines[0].split()[0])  # optional N on the first line
        start_idx = 1
    except Exception:
        ndecl = None

    seq = (lines[start_idx: start_idx+ndecl] if ndecl is not None else lines[start_idx:])
    for ln in seq:
        nums = re.findall(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?", ln)
        if len(nums) < 3: continue
        kx, ky, kz = map(float, nums[:3])
        w = int(round(float(nums[3]))) if len(nums) >= 4 else 1
        kpts.append((kx, ky, kz, w))
    return kpts if kpts else None

def _parse_mp_from_qe(text: str) -> Tuple[Tuple[int,int,int], Tuple[int,int,int], bool]:
    # gamma
    if re.search(r"(?im)^\s*K_POINTS\s+gamma\s*$", text):
        return (1,1,1), (0,0,0), True
    # automatic nx ny nz sx sy sz
    blk = _block_after(r"(?im)^\s*K_POINTS\s+automatic\s*$", text)
    if blk:
        nums = [int(x) for x in re.findall(r"[-+]?\d+", blk)]
        if len(nums) >= 6:
            nx,ny,nz,sx,sy,sz = nums[:6]
            grid  = (max(nx,1), max(ny,1), max(nz,1))
            shift = (sx,sy,sz)
            gamma = (grid==(1,1,1) and shift==(0,0,0))
            return grid, shift, gamma
        if len(nums) >= 3:
            nx,ny,nz = nums[:3]
            grid = (max(nx,1), max(ny,1), max(nz,1))
            gamma = (grid==(1,1,1))
            return grid, (0,0,0), gamma
    return (1,1,1), (0,0,0), True



def _mp_to_kpoints(nx:int, ny:int, nz:int, sx:int, sy:int, sz:int):
    """Generate fractional k-points for a Monkhorst–Pack grid.
       IMPORTANT: when n==1, the single point is exactly 0.0 (Γ), regardless of shift.
    """
    def coords(n, s):
        if n <= 1:
            return [0.0]                    # Γ only
        # QE convention for even grids:
        # shift=0 → i/n - 0.5 ; shift=1 → (i+0.5)/n - 0.5
        off = 0.5 if (s % 2) else 0.0
        return [((i + off)/n - 0.5) for i in range(n)]

    xs = coords(nx, sx); ys = coords(ny, sy); zs = coords(nz, sz)
    return [(x, y, z, 1) for x in xs for y in ys for z in zs]


def _infer_grid_from_kpoints(kpts: List[Tuple[float,float,float,int]], tol: float=1e-8):
    """Best-effort inference of (nx,ny,nz) and (sx,sy,sz) from an explicit regular grid."""
    xs = sorted(set(round(k[0]/tol)*tol for k in kpts))
    ys = sorted(set(round(k[1]/tol)*tol for k in kpts))
    zs = sorted(set(round(k[2]/tol)*tol for k in kpts))
    nx, ny, nz = len(xs), len(ys), len(zs)
    if nx*ny*nz != len(kpts):
        return None  # not a full rectilinear grid

    def infer_shift(coord_list, n):
        # For n==1: value ~0 -> shift 0
        if n == 1:
            return 0
        minc = coord_list[0]
        # Compare first point to -0.5 + (off)/n, where off=0 or 0.5
        # i=0 location is off/n - 0.5
        zero_pos = -0.5
        off0 = min(abs(minc - (0.0/n + zero_pos)), abs(minc - (0.5/n + zero_pos)))
        return 0 if off0 < abs(minc - (0.5/n + zero_pos)) else 1

    sx = infer_shift(xs, nx)
    sy = infer_shift(ys, ny)
    sz = infer_shift(zs, nz)
    return (nx,ny,nz), (sx,sy,sz)

# ---------- main writer ----------
def write_win_from_qe_in(
    qe_in: Path,
    seedname: str,
    out_dir: Path,
    *,
    num_wann: int = 8,
    num_bands: Optional[int] = None,                 # set > num_wann to disentangle
    projections: Iterable[str] = ("c: s; p", "h: s"),
    bands_plot: bool = False,
    dis_win: Optional[Tuple[float,float]] = None,
    dis_froz: Optional[Tuple[float,float]] = None,
    dis_num_iter: int = 1000,
    write_xyz: bool = True,
    kmesh_tol: float = 1e-6,
):
    text = Path(qe_in).read_text(errors="ignore")
    lat = _parse_cell_from_ibrav1(text)
    atoms = _parse_atoms(text, lat)

    # Prefer explicit QE list; else synthesize from automatic/gamma
    explicit = _parse_explicit_kpoints(text)
    mp_grid = None
    mp_shift = None
    gamma_only = False

    if explicit is None:
        (nx,ny,nz), (sx,sy,sz), gamma_only = _parse_mp_from_qe(text)
        explicit = _mp_to_kpoints(nx, ny, nz, sx, sy, sz)
        # Force gamma_only if it's truly Γ-only (sanity)
        gamma_only = (nx,ny,nz)==(1,1,1) and (sx,sy,sz)==(0,0,0)

    else:
        # Try to parse an accompanying 'automatic' to record mp_grid/mp_shift too
        grid_shift_gamma = _parse_mp_from_qe(text)
        if grid_shift_gamma is not None:
            mp_grid, mp_shift, gamma_only = grid_shift_gamma
        # If still None, try to infer from the explicit list
        if mp_grid is None:
            inf = _infer_grid_from_kpoints(explicit, tol=1e-8)
            if inf is not None:
                mp_grid, mp_shift = inf
                gamma_only = (mp_grid==(1,1,1) and mp_shift==(0,0,0))

    # Final fallbacks
    if mp_grid is None:
        mp_grid = (1,1,1)
    #if mp_shift is None:
    #    mp_shift = (0,0,0)

    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    win = out_dir / f"{seedname}.win"

    lines: List[str] = [
        f"num_wann = {num_wann}",
        f"mp_grid = {mp_grid[0]} {mp_grid[1]} {mp_grid[2]}",
        #f"mp_shift = {mp_shift[0]} {mp_shift[1]} {mp_shift[2]}",
        f"gamma_only = {'true' if gamma_only else 'false'}",
        f"kmesh_tol = {kmesh_tol:.1e}",
        f"bands_plot = {'true' if bands_plot else 'false'}",
        f"num_iter = {dis_num_iter}",
        f"write_xyz = {'true' if write_xyz else 'false'}",
        f"write_hr ={'true'}"
    ]
    if num_bands is not None:
        lines.append(f"num_bands = {int(num_bands)}")
    if dis_win is not None:
        lines += [f"dis_win_min = {dis_win[0]}", f"dis_win_max = {dis_win[1]}"]
    if dis_froz is not None:
        lines += [f"dis_froz_min = {dis_froz[0]}", f"dis_froz_max = {dis_froz[1]}"]

    # Always write explicit k-point list
    lines += ["", "begin kpoints"]
    lines += [f"{kx:.10f} {ky:.10f} {kz:.10f} {w:d}" for (kx,ky,kz,w) in explicit]
    lines += ["end kpoints", ""]

    # Geometry + projections
    lines += [
        "begin unit_cell_cart",
        "angstrom",
        *(f"{lat[i][0]:.10f} {lat[i][1]:.10f} {lat[i][2]:.10f}" for i in range(3)),
        "end unit_cell_cart",
        "",
        "begin atoms_cart",
        "angstrom",
        *(f"{s} {x:.10f} {y:.10f} {z:.10f}" for s,x,y,z in atoms),
        "end atoms_cart",
        "",
        "begin projections",
        *projections,
        "end projections",
        "",
    ]

    win.write_text("\n".join(lines))
    print(f"[ok] Wrote {win.name} with {len(explicit)} explicit k-points; mp_grid={mp_grid}, mp_shift={mp_shift}, gamma_only={gamma_only}")
    return win


In [24]:
# --- locate binaries ---
PW_BIN     = which_or_raise("pw.x")
PW2W90_BIN = which_or_raise("pw2wannier90.x", alt="pw2wannier90")
W90_BIN    = which_or_raise("wannier90.x", alt="wannier90")

# --- parse outdir/prefix, ensure absolute outdir ---
outdir_str, prefix_in = parse_outdir_prefix(qe_input, default_outdir="./")
# prefer absolute, consistent outdir
outdir_path = (Path(outdir_str).resolve() if not outdir_str.startswith("/") else Path(outdir_str))
# if outdir was "./", make it this workdir
if str(outdir_path) == "/.": outdir_path = workdir
outdir_path.mkdir(parents=True, exist_ok=True)

print(f"[info] prefix={prefix}  outdir={outdir_path}")
nscf_in = workdir / "nscf.in"
txt = qe_input.read_text(errors="ignore")

# --- 2) ensure we have a matching NSCF on Gamma (Gamma in .win above) ---
# Create NSCF input from your QE input by patching calculation and K_POINTS.

# force calculation='nscf', wf_collect=.true., and Gamma-only kpoints
txt = re.sub(r"(?i)calculation\s*=\s*'[^']+'", "calculation = 'nscf'", txt)
if "wf_collect" not in txt:
    txt = re.sub(r"(?is)&control", "&control\n   wf_collect = .true.,", txt, count=1)
# patch K_POINTS to Gamma
if re.search(r"(?i)K_POINTS", txt):
    txt = re.sub(r"(?is)K_POINTS\s+.*?\n", "K_POINTS gamma\n", txt)
else:
    txt += "\nK_POINTS gamma\n"
# ensure nbnd is at least num_wann
#if "nbnd" in txt.lower():
#    txt = re.sub(r"(?i)nbnd\s*=\s*\d+", f"nbnd = {max(num_bands_for_wann, infer_num_bands(qe_input))}", txt)
#else:
#    txt = re.sub(r"(?is)&system", f"&system\n   nbnd = {max(num_bands_for_wann, infer_num_bands(qe_input))},", txt, count=1)

nscf_in.write_text(txt)
print(f"[ok] wrote {nscf_in.name}")

# --- 1) write a minimal .win if missing (Γ-only) ---
#num_bands_for_wann = infer_num_bands(qe_input)
write_win_from_qe_in(Path("nscf.in"), "CH4", Path("."), num_wann=8, num_bands=12, projections=("c: s; p","h: s"),dis_win=(-5.0, 10.0),dis_froz=(-5.0, 0.5),dis_num_iter=1000,)


[info] prefix=CH4  outdir=/cache/home/am4655/qepy_eg/QCOMP/tmp
[ok] wrote nscf.in
[ok] Wrote CH4.win with 1 explicit k-points; mp_grid=(1, 1, 1), mp_shift=None, gamma_only=True


PosixPath('CH4.win')

In [25]:
# Run NSCF
print("[info] running pw.x (NSCF)...")
nscf_out = workdir / "nscf.out"
cp = run_cmd([PW_BIN, "-in", str(nscf_in)], cwd=workdir, check=False)
nscf_out.write_text((cp.stdout or "") + "\n" + (cp.stderr or ""))
print(f"[info] pw.x (NSCF) exited {cp.returncode}")
if cp.returncode != 0:
    print("[warn] NSCF had a non-zero exit; inspect nscf.out for details.")
# Verify prefix.save exists where pw2wannier90 will look
if not check_prefix_save(outdir_path, prefix):
    raise RuntimeError(f"{(outdir_path / (prefix + '.save'))} not found; ensure outdir/prefix and permissions are correct.")

# --- 3) wannier90 -pp to create seed.nnkp for Γ ---
print("[info] running wannier90 -pp ...")
pp = run_cmd([W90_BIN, "-pp", seedname], cwd=workdir, check=False)
(workdir / f"{seedname}.pp.log").write_text((pp.stdout or "") + "\n" + (pp.stderr or ""))
print(f"[info] wannier90 -pp exit {pp.returncode}")
if (workdir / f"{seedname}.nnkp").exists():
    print(f"[ok] found {(workdir / f'{seedname}.nnkp').name}")
else:
    raise RuntimeError(f"{seedname}.nnkp not created; check {seedname}.pp.log")

# --- 4) pw2wannier90: point to the same outdir/prefix, no shell redirection ---
pw2w90_in = textwrap.dedent(f"""
&inputpp
   outdir   = '{str(outdir_path)}'
   prefix   = '{prefix}'
   seedname = '{seedname}'
   write_amn = .true.
   write_mmn = .true.
   write_unk = .false.
/
""").strip()
(workdir / "pw2wannier90.in").write_text(pw2w90_in)

print("[info] running pw2wannier90.x ...")
pw2 = run_cmd([PW2W90_BIN], cwd=workdir, stdin_text=pw2w90_in, check=False)
(workdir / "pw2wannier90.out").write_text((pw2.stdout or "") + "\n" + (pw2.stderr or ""))
print(f"[info] pw2wannier90 exit {pw2.returncode}")
if pw2.returncode != 0:
    # Show the last 60 lines to pinpoint the issue
    tail = "\n".join((pw2.stdout or pw2.stderr or "").splitlines()[-60:])
    raise RuntimeError("pw2wannier90 failed.\n--- tail ---\n" + tail)

[info] running pw.x (NSCF)...
[info] pw.x (NSCF) exited 0
[check] looking for /cache/home/am4655/qepy_eg/QCOMP/tmp/CH4.save
[info] running wannier90 -pp ...
[info] wannier90 -pp exit 0
[ok] found CH4.nnkp
[info] running pw2wannier90.x ...
[info] pw2wannier90 exit 0


In [26]:
# --- 5) final wannierization ---
print("[info] running wannier90 ...")
w90 = run_cmd([W90_BIN, seedname], cwd=workdir, check=False)
(workdir / f"{seedname}.w90.out").write_text((w90.stdout or "") + "\n" + (w90.stderr or ""))
print(f"[info] wannier90 exit {w90.returncode}")

hr = workdir / f"{seedname}_hr.dat"
if hr.exists():
    print(f"[ok] Found {hr}")
else:
    raise RuntimeError(f"{hr} not found — check pw2wannier90.out and {seedname}.w90.out")

[info] running wannier90 ...
[info] wannier90 exit 0
[ok] Found /cache/home/am4655/qepy_eg/QCOMP/CH4_hr.dat


In [27]:
from dataclasses import dataclass
import numpy as np

@dataclass
class WannierTB:
    Rvecs: np.ndarray         # (nR, 3) integer lattice translations
    degeneracies: np.ndarray  # (nR,) symmetry weights
    H_R: np.ndarray           # (nR, nW, nW) complex, eV

def _is_int(s):
    try: int(s); return True
    except: return False

def _is_float(s):
    try: float(s); return True
    except: return False

def read_wannier90_hr(hr_path) -> WannierTB:
    # read all non-empty lines
    raw = [ln.strip() for ln in open(hr_path, "r") if ln.strip()]
    # header
    title = raw[0]               # unused
    nw = int(raw[1])
    nR = int(raw[2])

    # degeneracy block: 15 per line (last line shorter)
    degs, i = [], 3
    while len(degs) < nR:
        degs.extend([int(x) for x in raw[i].split()])
        i += 1
    degeneracies = np.array(degs[:nR], dtype=int)

    # storage
    Rvecs = np.zeros((nR, 3), dtype=int)
    H_R   = np.zeros((nR, nw, nw), dtype=np.complex128)

    # helper to map/append an R-vector to index
    def idx_for_R(Rxyz):
        nonlocal Rvecs, H_R, degeneracies
        matches = np.where((Rvecs == Rxyz).all(axis=1))[0]
        if len(matches): return matches[0]
        # find first empty row
        zeros = np.where(~Rvecs.any(axis=1))[0]
        if len(zeros):
            j = zeros[0]
            Rvecs[j] = Rxyz
            return j
        # otherwise grow arrays (rare: if hr lists more unique R than nR header)
        Rvecs = np.vstack([Rvecs, Rxyz[None, :]])
        H_R   = np.concatenate([H_R, np.zeros((1, nw, nw), dtype=np.complex128)], axis=0)
        degeneracies = np.pad(degeneracies, (0,1), constant_values=1)  # safe default
        return Rvecs.shape[0] - 1

    # parse all data lines flexibly
    for ln in raw[i:]:
        if ln.startswith("#"):   # ignore comments
            continue
        tok = ln.split()

        # Accept any of these forms:
        #  A) ir Rx Ry Rz  i  j  Re  Im    (len>=8, first two are ints)
        #  B)    Rx Ry Rz  i  j  Re  Im    (len>=7, first three ints)
        #  C)    Rx Ry Rz  i  j  Re        (len==6, no Im -> 0)
        R = None; iorb = jorb = None; ReH = ImH = None

        # Try A: with leading ir
        if len(tok) >= 8 and _is_int(tok[0]) and _is_int(tok[1]) and _is_int(tok[2]) and _is_int(tok[3]) and _is_int(tok[4]) and _is_int(tok[5]) and _is_float(tok[6]):
            # ir, Rx, Ry, Rz, i, j, Re, [Im]
            Rx, Ry, Rz = int(tok[1]), int(tok[2]), int(tok[3])
            iorb, jorb = int(tok[4]) - 1, int(tok[5]) - 1
            ReH = float(tok[6])
            ImH = float(tok[7]) if len(tok) >= 8 and _is_float(tok[7]) else 0.0

        # Try B: no leading ir
        elif len(tok) >= 7 and _is_int(tok[0]) and _is_int(tok[1]) and _is_int(tok[2]) and _is_int(tok[3]) and _is_int(tok[4]) and _is_float(tok[5]):
            # Rx, Ry, Rz, i, j, Re, [Im]
            Rx, Ry, Rz = int(tok[0]), int(tok[1]), int(tok[2])
            iorb, jorb = int(tok[3]) - 1, int(tok[4]) - 1
            ReH = float(tok[5])
            ImH = float(tok[6]) if len(tok) >= 7 and _is_float(tok[-1]) and len(tok) >= 7 else 0.0

        # Try C: 6 tokens (no Im)
        elif len(tok) == 6 and all(_is_int(x) for x in tok[:5]) and _is_float(tok[5]):
            Rx, Ry, Rz = int(tok[0]), int(tok[1]), int(tok[2])
            iorb, jorb = int(tok[3]) - 1, int(tok[4]) - 1
            ReH = float(tok[5])
            ImH = 0.0

        else:
            # Last attempt: detect first 3 ints as R, next 2 ints as (i,j), then the rest as floats
            ints = [k for k,x in enumerate(tok) if _is_int(x)]
            if len(tok) >= 6 and len(ints) >= 5:
                Rx, Ry, Rz = int(tok[0]), int(tok[1]), int(tok[2])
                iorb, jorb = int(tok[3]) - 1, int(tok[4]) - 1
                floats = [float(x) for x in tok[5:]]
                if len(floats) == 1:
                    ReH, ImH = floats[0], 0.0
                elif len(floats) >= 2:
                    ReH, ImH = floats[0], floats[1]
                else:
                    raise ValueError(f"Cannot parse energies in line: {ln}")
            else:
                raise ValueError(f"Unexpected hr line format: {ln}")

        R = np.array([Rx, Ry, Rz], dtype=int)
        ir = idx_for_R(R)
        H_R[ir, iorb, jorb] += (ReH + 1j*ImH)

    return WannierTB(Rvecs=Rvecs, degeneracies=degeneracies, H_R=H_R)


In [31]:
def fermionic_from_tb(H: np.ndarray, U: float = 0.0, V_nn: float = 0.0,
                      nn_pairs: List[Tuple[int,int]] | None = None) -> FermionicOp:
    """
    Spinful TB at a single k (default Γ). 2*nW spin-orbitals: (i,↑),(i,↓).
    One-body: Σ_{ij,σ} H_ij a†_{iσ} a_{jσ}
    Two-body: U Σ_i n_{i↑}n_{i↓} + V_nn Σ_<i,j> n_i n_j
    """
    nW = H.shape[0]
    labels: Dict[str, complex] = {}

    def add1(i, j, c):
        labels[f"+_{2*i} -_{2*j}"] = labels.get(f"+_{2*i} -_{2*j}", 0) + c
        labels[f"+_{2*i+1} -_{2*j+1}"] = labels.get(f"+_{2*i+1} -_{2*j+1}", 0) + c

    def addU(i, c):
        labels[f"+_{2*i} -_{2*i} +_{2*i+1} -_{2*i+1}"] = labels.get(
            f"+_{2*i} -_{2*i} +_{2*i+1} -_{2*i+1}", 0) + c

    def addV(i, j, c):
        for si in (0,1):
            for sj in (0,1):
                a = 2*i+si; b = 2*j+sj
                labels[f"+_{a} -_{a} +_{b} -_{b}"] = labels.get(
                    f"+_{a} -_{a} +_{b} -_{b}", 0) + c

    for i in range(nW):
        for j in range(nW):
            if abs(H[i, j]) > 1e-12:
                add1(i, j, complex(H[i, j]))

    if U != 0.0:
        for i in range(nW):
            addU(i, U)

    if V_nn != 0.0 and nn_pairs:
        for (i, j) in nn_pairs:
            addV(i, j, V_nn)

    return FermionicOp(labels, num_spin_orbitals=2*nW)

def kspace_hamiltonian(tb: WannierTB, k: Tuple[float, float, float]) -> np.ndarray:
    R = tb.Rvecs
    Hk = np.zeros(tb.H_R.shape[1:], dtype=np.complex128)
    phase = np.exp(2j * np.pi * (R @ np.array(k)))
    for ir in range(R.shape[0]):
        Hk += tb.degeneracies[ir] * phase[ir] * tb.H_R[ir]
    return Hk

def particle_number_operator(num_spin_orbitals: int) -> FermionicOp:
    labels: Dict[str, complex] = {}
    for p in range(num_spin_orbitals):
        labels[f"+_{p} -_{p}"] = labels.get(f"+_{p} -_{p}", 0) + 1.0
    return FermionicOp(labels, num_spin_orbitals=num_spin_orbitals)

def vqe_ground_energy(fermi_op: FermionicOp, n_particles: int | None,
                      penalty_coef: float = 0.0,
                      ansatz_layers: int = 2,
                      shots: int | None = None) -> Tuple[float, SparsePauliOp]:
    mapper = JordanWignerMapper()
    qubit_op = mapper.map(fermi_op)

    # Optional particle-number penalty (N̂ - N)^2
    if n_particles is not None and penalty_coef > 0:
        N_op_f = particle_number_operator(fermi_op.num_spin_orbitals)
        N_op_q = mapper.map(N_op_f)
        N_sq = (N_op_q @ N_op_q).simplify()
        penalty = (N_sq * penalty_coef
                   - (N_op_q * (2 * penalty_coef * n_particles))
                   + SparsePauliOp.from_list([("I"*N_op_q.num_qubits, penalty_coef*(n_particles**2))]))
        qubit_op = (qubit_op + penalty).simplify()

    ansatz = EfficientSU2(num_qubits=qubit_op.num_qubits, reps=ansatz_layers, entanglement="linear")
    optimizer = SPSA(maxiter=200)
    estimator = Estimator(options={"shots": shots} if shots else None)
    vqe = VQE(estimator=estimator, ansatz=ansatz, optimizer=optimizer)
    result = vqe.compute_minimum_eigenvalue(qubit_op)
    e = float(np.real(result.eigenvalue))
    return e, qubit_op


In [29]:
drv.get_occupation_numbers()

array([2., 2., 2., 2., 0., 0., 0., 0., 0., 0., 0., 0.])

In [32]:
# --- Knobs ---
k = (0.0, 0.0, 0.0)     # Γ point
U = 0.0                 # onsite Hubbard-U in eV (set > 0 to include correlation)
V_nn = 0.0              # nearest-neighbor density-density; requires nn_pairs
nn_pairs = None        # e.g., [(0,1), (1,2)]
electrons = 10       # integer to softly enforce particle number via penalty (None = no constraint)
penalty = 5.0           # coefficient for (N̂ - N)^2
ansatz_layers = 2
shots = None            # None = exact expectations; set e.g. 8192 to sample

# --- Build and solve ---
tb = read_wannier90_hr(hr)
Hk = kspace_hamiltonian(tb, k)
fop = fermionic_from_tb(Hk, U=U, V_nn=V_nn, nn_pairs=nn_pairs)
energy, qubit_op = vqe_ground_energy(fop, n_particles=electrons, penalty_coef=penalty,
                                     ansatz_layers=ansatz_layers, shots=shots)

print("Qubits:", qubit_op.num_qubits)
print("Hamiltonian terms:", len(qubit_op))
print("VQE Ground-state energy (eV):", energy)


/tmp/ipykernel_40707/285270857.py:74: DeprecationWarning: The class ``qiskit.primitives.estimator.Estimator`` is deprecated as of qiskit 1.2. It will be removed no earlier than 3 months after the release date. All implementations of the `BaseEstimatorV1` interface have been deprecated in favor of their V2 counterparts. The V2 alternative for the `Estimator` class is `StatevectorEstimator`.
  estimator = Estimator(options={"shots": shots} if shots else None)


Qubits: 16
Hamiltonian terms: 249
VQE Ground-state energy (eV): 14.186336839557


In [138]:
from qiskit_algorithms import VQE
from qiskit.primitives import Estimator
from qiskit.circuit.library import EfficientSU2
from qiskit_nature.second_q.algorithms import ExcitedStatesEigensolver, QEOM

ansatz = EfficientSU2(qubit_op.num_qubits, reps=2, entanglement="linear")
opt = SPSA(maxiter=200)
vqe = VQE(Estimator(), ansatz, opt)
# Ground state
gs_result = vqe.compute_minimum_eigenvalue(qubit_op)

# Excited states via QEOM
qeom = ExcitedStatesEigensolver(vqe, QEOM())
es_result = qeom.evaluate(qubit_op)
print("Energies:", es_result.raw_results.computed_eigenvalues)


/tmp/ipykernel_32516/3121153773.py:8: DeprecationWarning: The class ``qiskit.primitives.estimator.Estimator`` is deprecated as of qiskit 1.2. It will be removed no earlier than 3 months after the release date. All implementations of the `BaseEstimatorV1` interface have been deprecated in favor of their V2 counterparts. The V2 alternative for the `Estimator` class is `StatevectorEstimator`.
  vqe = VQE(Estimator(), ansatz, opt)


TypeError: __init__() missing 2 required positional arguments: 'ground_state_solver' and 'estimator'